# ⛑️ KY表 AI作成ツール
**セル①→②の順に実行してください（▶ボタン）**

In [ ]:
# ① パッケージのインストール（最初の1回だけ）
!pip install anthropic flask flask-cors -q

In [ ]:
# ② APIキーをここに貼り付けて▶を押すと起動します
ANTHROPIC_API_KEY = "sk-ant-api03-ここに貼り付け"

# ------- 以下は変更不要 -------
import anthropic, threading, json, re
from flask import Flask, request, jsonify, Response
from flask_cors import CORS
from google.colab.output import eval_js

app = Flask(__name__)
CORS(app)
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

HTML = """
<!DOCTYPE html>
<html lang='ja'>
<head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1.0'>
<title>KY表 AI作成ツール</title>
<style>
  * { margin:0; padding:0; box-sizing:border-box; }
  body { font-family:'Hiragino Sans','Yu Gothic UI',sans-serif; background:#f0f4f8; color:#333; min-height:100vh; }
  .accent-bar { height:4px; background:linear-gradient(90deg,#0096C7,#FF6B35); }
  header { background:#0096C7; color:white; padding:0 32px; height:64px; display:flex; align-items:center; box-shadow:0 2px 8px rgba(0,0,0,.15); }
  .logo { font-size:18px; font-weight:bold; }
  .logo span { font-size:11px; display:block; font-weight:normal; opacity:.8; margin-top:2px; }
  .container { max-width:760px; margin:36px auto; padding:0 16px; }
  h1 { font-size:22px; color:#0077A8; margin-bottom:6px; }
  .subtitle { font-size:13px; color:#888; margin-bottom:28px; }
  .card { background:white; border-radius:12px; padding:28px; box-shadow:0 2px 12px rgba(0,0,0,.08); margin-bottom:24px; }
  .section-title { font-size:13px; font-weight:bold; color:#0096C7; border-left:4px solid #FF6B35; padding-left:10px; margin-bottom:18px; }
  .form-row { display:flex; gap:16px; margin-bottom:16px; }
  .form-group { display:flex; flex-direction:column; gap:6px; flex:1; }
  label { font-size:12px; font-weight:bold; color:#555; }
  input,select,textarea { border:1.5px solid #ddd; border-radius:8px; padding:10px 12px; font-size:14px; font-family:inherit; transition:border-color .2s; outline:none; }
  input:focus,select:focus,textarea:focus { border-color:#0096C7; }
  textarea { resize:vertical; min-height:80px; }
  .btn-suggest { width:100%; padding:14px; background:#FF6B35; color:white; border:none; border-radius:10px; font-size:16px; font-weight:bold; cursor:pointer; transition:background .2s; margin-top:4px; }
  .btn-suggest:hover { background:#e55a24; }
  .btn-suggest:disabled { background:#ccc; cursor:not-allowed; }
  .result-card { background:white; border-radius:12px; padding:28px; box-shadow:0 2px 12px rgba(0,0,0,.08); display:none; }
  .result-card.visible { display:block; }
  .ky-pair { display:grid; grid-template-columns:1fr 1fr; gap:16px; margin-bottom:16px; }
  .ky-box { background:#f8fbff; border:1.5px solid #c8e6f5; border-radius:8px; padding:14px; }
  .ky-box.action { background:#fff8f5; border-color:#ffd5c2; }
  .ky-label { font-size:11px; font-weight:bold; color:#0096C7; margin-bottom:8px; display:flex; align-items:center; gap:6px; }
  .ky-box.action .ky-label { color:#FF6B35; }
  .ky-num { display:inline-flex; align-items:center; justify-content:center; width:18px; height:18px; border-radius:50%; background:#0096C7; color:white; font-size:10px; font-weight:bold; }
  .ky-box.action .ky-num { background:#FF6B35; }
  .ky-box textarea { border:none; background:transparent; padding:0; font-size:13px; width:100%; color:#333; min-height:60px; }
  .ky-box textarea:focus { outline:none; }
  .goal-box { background:linear-gradient(135deg,#e8f7ff,#fff0ea); border:2px solid #0096C7; border-radius:10px; padding:16px 20px; margin-top:4px; }
  .goal-label { font-size:11px; font-weight:bold; color:#0096C7; margin-bottom:8px; }
  .goal-box textarea { border:none; background:transparent; padding:0; font-size:15px; font-weight:bold; width:100%; color:#0077A8; min-height:36px; }
  .goal-box textarea:focus { outline:none; }
  .loading { display:none; text-align:center; padding:20px; color:#0096C7; font-size:14px; gap:10px; align-items:center; justify-content:center; }
  .loading.visible { display:flex; }
  .spinner { width:20px; height:20px; border:3px solid #c8e6f5; border-top-color:#0096C7; border-radius:50%; animation:spin .8s linear infinite; }
  @keyframes spin { to { transform:rotate(360deg); } }
  .error-box { background:#fff0f0; border:1.5px solid #ffb3b3; border-radius:8px; padding:12px 16px; color:#cc0000; font-size:13px; display:none; margin-top:12px; }
  .error-box.visible { display:block; }
  footer { text-align:center; padding:24px; font-size:12px; color:#aaa; }
  @media(max-width:560px) { .ky-pair { grid-template-columns:1fr; } .form-row { flex-direction:column; } }
</style>
</head>
<body>
<div class='accent-bar'></div>
<header><div class='logo'>⛑️ KY表 AI作成ツール<span>公民連携沖縄株式会社｜造園・公園管理</span></div></header>
<div class='container'>
  <h1>KY活動 AI提案フォーム</h1>
  <p class='subtitle'>作業内容を入力すると、AIが危険ポイント・対策・安全目標を提案します</p>
  <div class='card'>
    <div class='section-title'>📋 基本情報を入力</div>
    <div class='form-row'>
      <div class='form-group' style='flex:0 0 80px'><label>月</label><input type='number' id='month' min='1' max='12'></div>
      <div class='form-group' style='flex:0 0 80px'><label>日</label><input type='number' id='day' min='1' max='31'></div>
      <div class='form-group' style='flex:0 0 120px'><label>天候</label><select id='weather'><option>晴れ</option><option>曇り</option><option>雨</option><option>荒天</option></select></div>
      <div class='form-group' style='flex:0 0 100px'><label>作業員数</label><input type='number' id='workers' min='1'></div>
    </div>
    <div class='form-group' style='margin-bottom:16px'>
      <label>グループの作業内容 <span style='color:#FF6B35'>*</span></label>
      <textarea id='workContent' placeholder='例：公園内の樹木剪定作業（高さ3〜5mの高木5本）'></textarea>
    </div>
    <div class='form-row'>
      <div class='form-group'><label>リーダー名</label><input type='text' id='leader' placeholder='上原'></div>
      <div class='form-group'><label>参加者氏名</label><input type='text' id='members' placeholder='上原、田中、山田'></div>
    </div>
    <button class='btn-suggest' id='suggestBtn' onclick='suggest()'>🤖 AIに危険ポイントを提案してもらう</button>
    <div class='error-box' id='errorBox'></div>
  </div>
  <div class='loading' id='loading'><div class='spinner'></div>AIが危険ポイントを考えています...</div>
  <div class='result-card' id='resultCard'>
    <div class='section-title'>⚠️ AI提案結果（編集できます）</div>
    <div class='ky-pair'>
      <div class='ky-box'><div class='ky-label'><span class='ky-num'>①</span>危険のポイント</div><textarea id='danger1'></textarea></div>
      <div class='ky-box action'><div class='ky-label'><span class='ky-num'>①</span>私達はこうする</div><textarea id='action1'></textarea></div>
    </div>
    <div class='ky-pair'>
      <div class='ky-box'><div class='ky-label'><span class='ky-num'>②</span>危険のポイント</div><textarea id='danger2'></textarea></div>
      <div class='ky-box action'><div class='ky-label'><span class='ky-num'>②</span>私達はこうする</div><textarea id='action2'></textarea></div>
    </div>
    <div class='goal-box'><div class='goal-label'>🎯 本日の安全目標</div><textarea id='safetyGoal'></textarea></div>
  </div>
</div>
<footer>© 2026 公民連携沖縄株式会社</footer>
<script>
const today = new Date();
document.getElementById('month').value = today.getMonth()+1;
document.getElementById('day').value = today.getDate();
async function suggest() {
  const workContent = document.getElementById('workContent').value.trim();
  if (!workContent) { showError('作業内容を入力してください'); return; }
  const btn = document.getElementById('suggestBtn');
  const loading = document.getElementById('loading');
  const resultCard = document.getElementById('resultCard');
  const errorBox = document.getElementById('errorBox');
  btn.disabled = true;
  loading.classList.add('visible');
  resultCard.classList.remove('visible');
  errorBox.classList.remove('visible');
  try {
    const res = await fetch('/api/suggest', {
      method:'POST',
      headers:{'Content-Type':'application/json'},
      body:JSON.stringify({workContent, weather:document.getElementById('weather').value, workers:document.getElementById('workers').value})
    });
    const data = await res.json();
    if (!res.ok) { showError(data.error || 'エラーが発生しました'); return; }
    document.getElementById('danger1').value = data.danger1||'';
    document.getElementById('action1').value = data.action1||'';
    document.getElementById('danger2').value = data.danger2||'';
    document.getElementById('action2').value = data.action2||'';
    document.getElementById('safetyGoal').value = data.safetyGoal||'';
    resultCard.classList.add('visible');
    resultCard.scrollIntoView({behavior:'smooth',block:'start'});
  } catch(e) { showError('通信エラーが発生しました。'); }
  finally { btn.disabled=false; loading.classList.remove('visible'); }
}
function showError(msg) {
  const e = document.getElementById('errorBox');
  e.textContent = '⚠️ ' + msg;
  e.classList.add('visible');
}
</script>
</body></html>
"""

@app.route('/')
def index():
    return Response(HTML, content_type='text/html; charset=utf-8')

@app.route('/api/suggest', methods=['POST'])
def suggest():
    data = request.get_json()
    work_content = (data.get('workContent') or '').strip()
    if not work_content:
        return jsonify({'error': '作業内容を入力してください'}), 400
    weather = (data.get('weather') or '晴れ').strip()
    workers = data.get('workers') or ''
    prompt = f"""造園・公園管理の現場でのKY（危険予知）活動の内容を考えてください。

作業内容: {work_content}
天候: {weather}
作業員数: {str(workers)+'名' if workers else '不明'}

以下のJSON形式のみで返してください（説明文は不要）:
{{"danger1":"危険のポイント①","action1":"私達はこうする①","danger2":"危険のポイント②","action2":"私達はこうする②","safetyGoal":"本日の安全目標"}}"""
    try:
        message = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=1024,
            messages=[{'role': 'user', 'content': prompt}]
        )
        text = message.content[0].text.strip()
        m = re.search(r'\{[\s\S]*\}', text)
        if not m:
            return jsonify({'error': 'AI応答の解析に失敗しました'}), 500
        return jsonify(json.loads(m.group()))
    except Exception as e:
        return jsonify({'error': str(e)}), 500

t = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False))
t.daemon = True
t.start()

url = eval_js("google.colab.kernel.proxyPort(5000)")
print(f"\n✅ 起動しました！\n\n🔗 以下のURLをブラウザで開いてください:\n{url}\n")